<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBD-EN/blob/main/08_agentic_ai/agentic_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -Uqqq numpy>=1.24.0 langchain langchain-community langchain-experimental langchain-classic transformers torch accelerate bitsandbytes gradio langchain-huggingface faiss-cpu sentence-transformers

In [ ]:
import torch
import warnings
warnings.filterwarnings('ignore') # Suprime warnings de cuantización para limpiar la consola

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_experimental.tools import PythonREPLTool
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool

# ==========================================
# 1. THE BRAIN (LLM Configuration)
# ==========================================
print("1. Loading physical weights (TinyLlama 1.1B)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.01,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)

# ==========================================
# 2. TOOLS (Vector DB & Python)
# ==========================================
print("2. Formatting Memory and Logic tools...")
planet_facts = [
    "The fictional planet 'Aethelgard' has 4 moons.",
    "Aethelgard's atmosphere is composed of 70% Helium and 30% Neon.",
    "The core temperature of Aethelgard is exactly 12400 degrees Celsius.",
    "Aethelgard's gravity is 2.5 times stronger than Earth's gravity."
]
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(planet_facts, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

def query_database(query: str) -> str:
    docs = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])

rag_tool = Tool(
    name="Aethelgard_Database",
    description="Search for facts about Aethelgard. Input should be a single search query string.",
    func=query_database
)

python_tool = PythonREPLTool()
tools = [rag_tool, python_tool]

# ==========================================
# 3. ARCHITECTURE (ReAct Loop)
# ==========================================
print("3. Compiling ReAct State Machine...")
template = """Answer the query. You have these tools:
{tools}

Format strictly:
Question: input
Thought: what to do
Action: tool name from [{tool_names}]
Action Input: tool input
Observation: tool result
... (repeat Thought/Action/Observation)
Thought: I have the answer
Final Answer: the answer

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)
agent = create_react_agent(llm, tools, prompt)

# verbose=True es el núcleo pedagógico aquí: imprime el monólogo en verde/azul por consola
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=2
)



In [ ]:
# ==========================================
# 4. COMMAND LINE INTERFACE
# ==========================================
print("\n" + "="*60)
print("🤖 Agentic CLI Ready. (Type 'exit' to terminate)")
print("="*60)

while True:
    user_input = input("\n👤 Question: ")

    if user_input.lower() in ['exit', 'quit']:
        print("Terminating hardware session.")
        break

    if not user_input.strip():
        continue

    print("\n🧠 Executing Agent Loop...\n" + "-"*40)
    try:
        # La ejecución bloqueará la terminal y escupirá los pasos en tiempo real
        result = agent_executor.invoke({"input": user_input})

        print("-" * 40)
        print(f"✅ Extracted Answer: {result.get('output', 'N/A')}")

    except Exception as e:
        print(f"\n❌ Critical Fault: {str(e)}")

# 🧠 Lab Session: Agentic AI with Python Logic & Reasoning

**Objective:**
In this final session, we build an **Autonomous Agent**. Unlike a chatbot that just talks, an Agent can *do things*. We will give our model access to a **Python Interpreter**. This allows the AI to write code to solve logic puzzles, perform complex math, or process data, effectively giving it the capabilities of a Data Scientist.

**The Architecture:**
1.  **Brain:** TinyLlama (Quantized) decides *what* to do.
2.  **Tool A (Memory):** A Vector Database (RAG) for specific facts about our fictional planet.
3.  **Tool B (Logic):** A Python REPL (Read-Eval-Print Loop) to execute code.
4.  **Interface:** Gradio to visualize the Agent's "Thought Process".

## 1. Environment Setup

We need `langchain-experimental` to access the Python execution tools safely.

In [1]:
# @title 🛠️ Install Libraries
!pip install -Uqqq --force-reinstall numpy>=1.24.0 langchain langchain-community langchain-experimental langchainhub transformers torch accelerate bitsandbytes gradio langchain-huggingface
print("Agent Environment installed! NOW: Go to 'Runtime' -> 'Restart session' to apply changes.")

ERROR: Operation cancelled by user
Agent Environment installed! NOW: Go to 'Runtime' -> 'Restart session' to apply changes.


## 2. The Brain

We load the model in 4-bit mode to fit comfortably in the free Colab GPU. We set `temperature=0.01` because we need the model to be precise when calling tools, not creative.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline

# 1. 4-Bit Quantization Config (Efficiency First)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading {model_id} as the Agent's Brain...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 2. Text Generation Pipeline
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512, # Agents need space to "think"
    temperature=0.01,   # Precision over creativity
    repetition_penalty=1.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
print("Agent Brain Ready!")

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 as the Agent's Brain...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'repetition_penalty', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Agent Brain Ready!


## 3. Creating the Tools (Hands & Calculator)

We will give the Agent two superpowers:
1.  **Planetary Knowledge (RAG):** To retrieve facts it wasn't trained on.
2.  **Python Interpreter:** To execute code for math and logic.

In [2]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

# 1. Definimos la plantilla (El "First Principle" del razonamiento ReAct)
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

# 2. Creamos la función de decisión (El Agente)
agent = create_react_agent(llm, tools, prompt)

# 3. Inicializamos el motor de ejecución (El bucle)
# AQUÍ ES DONDE UTILIZAMOS AgentExecutor CORRECTAMENTE
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5 # Evita bucles infinitos si el modelo alucina
)

print("Agent Initialized. Ready to Reason and Act.")

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (/usr/local/lib/python3.12/dist-packages/langchain/agents/__init__.py)

## 4. Initializing the Agent (The Orchestrator)

We use the **ReAct (Reason + Act)** framework. The agent will loop through:
* **Thought:** "What should I do?"
* **Action:** "Run this Python code."
* **Observation:** "Here is the result of the code."
* **Final Answer:** "Here is the summary."

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

# 1. Definimos la plantilla base (El "First Principle" del razonamiento ReAct)
template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)

# 2. Creamos la función de decisión (El Agente)
agent = create_react_agent(llm, tools, prompt)

# 3. Inicializamos el motor de ejecución (El bucle)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

print("Agent Initialized. Ready to Reason and Act.")

In [ ]:
from langchain.agents.react.agent import create_react_agent
from langchain.agents.agent import AgentExecutor
from langchain import hub

# 1. Obtener el prompt del hub
prompt = hub.pull("hwchase17/react")

# 2. Construir el agente ReAct
agent_runnable = create_react_agent(llm, tools, prompt)

# 3. Crear el AgentExecutor
agent = AgentExecutor(
    agent=agent_runnable,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

print("Agent Initialized using create_react_agent. Ready to Reason and Act.")

## 5. Testing the Logic

Let's verify that the Python tool is working by asking something Llama-2 cannot do purely by text prediction (like iterating through a sequence).

In [ ]:
# Test: Fibonacci Sequence
# The LLM should write a Python loop to solve this.
query = "Generate the first 10 numbers of the Fibonacci sequence and calculate their sum."

print(f"User Query: {query}")
print("-" * 40)
result = agent.run(query)
print("-" * 40)
print(f"Final Answer: {result}")

## 6. The Interface (Gradio)

We will build a Chat Interface that reveals the **"Brain's Monologue"**. We capture the system logs (stdout) to show the user exactly how the Agent decided to write Python code or check the database.

In [ ]:
import gradio as gr
import sys
from io import StringIO

def solve_with_reasoning(message, history):
    # 1. Capture the "Thinking Process" (Stdout)
    old_stdout = sys.stdout
    sys.stdout = mystdout = StringIO()

    try:
        # 2. Run the Agent
        result = agent.run(message)
    except Exception as e:
        result = f"I encountered an error: {str(e)}"

    # 3. Restore Stdout & Get Logs
    sys.stdout = old_stdout
    reasoning_logs = mystdout.getvalue()

    # 4. Format Output for the UI
    # We use Markdown to make the code/logs look like a terminal
    formatted_output = (
        f"🧠 **Agent Thought Process:**\n"
        f"```bash\n{reasoning_logs}\n```\n\n"
        f"✅ **Final Answer:**\n{result}"
    )
    return formatted_output

# Create the Gradio Interface
demo = gr.ChatInterface(
    fn=solve_with_reasoning,
    title="🤖 Agentic AI: The Python Coder",
    description="Ask me complex questions. I can use a Database or write Python code to answer.",
    examples=[
        "What is 30% of the population of Proxima Centauri b?",
        "Create a list of numbers from 1 to 10, square them, and find the average.",
        "How many seconds does it take for light to travel from Earth to Proxima b? (Use Python)",
        "Sort this list alphabetically: [Zebra, Apple, Mango, Delta]"
    ],
    theme="glass"
)

demo.launch(share=True, debug=True)